# Phase 4 — Time-Aware Train/Validation Split
## Transaction Fraud Risk Engine

**Objective:** Split the Phase 2 engineered feature dataset into training and validation sets based on chronological order (`TransactionDT`), rather than a random split. This mirrors how the model will actually be used in production — trained on past transactions, evaluated on future ones it has never seen — and avoids the data leakage that a random split would introduce on time-ordered data.

**Input data:** `data/processed/engineered_features.csv` — the Phase 2 output (590,540 rows, 14 columns), statistically validated in Phase 3.

**Structure of this notebook:** Sort by time → determine the split cutoff → verify zero time overlap between sets → save the resulting train/validation sets for use in Phase 5.

## 4.1 Chronological Split

Sort all transactions by `TransactionDT` and split into training (earliest 80%) and validation (most recent 20%) sets, using a strict time cutoff rather than a random split. This ensures the validation set only ever contains transactions that occurred after every transaction in the training set — mirroring how the model would actually be deployed and evaluated in production.

In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/engineered_features.csv")
df = df.sort_values(by="TransactionDT").reset_index(drop=True)
print(f"Min Seconds: {df['TransactionDT'].min()} | Max Seconds: {df['TransactionDT'].max()}")

train_ratio = 0.80
split_index = int(len(df) * train_ratio)
print(f"Split index: {split_index}")
print("-"*20)
split_threshold_dt = df.loc[split_index, "TransactionDT"]
print(f"Split Threshold TransactionDT: {split_threshold_dt}")

train_df = df[df["TransactionDT"] <= split_threshold_dt].reset_index(drop=True)
test_df = df[df["TransactionDT"] > split_threshold_dt].reset_index(drop=True)

print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")
print("Train max TransactionDT:", train_df["TransactionDT"].max())
print("Test min TransactionDT:", test_df["TransactionDT"].min())

Min Seconds: 86400 | Max Seconds: 15811131
Split index: 472432
--------------------
Split Threshold TransactionDT: 12192900
Train: (472433, 14)
Test: (118107, 14)
Train max TransactionDT: 12192900
Test min TransactionDT: 12192911


### Observations:

- Dataset spans `TransactionDT` values from 86,400 to 15,811,131 (~183 days / ~6 months of transaction history).
- 80/20 split by row count produced: **Train = 472,433 rows, Test = 118,107 rows** — closely matching the intended ratio (a 1-row difference from the exact 80% mark occurred because multiple rows shared the exact threshold `TransactionDT` value; all were kept in training via `<=`, which is the safer direction — it never lets a boundary row leak into validation).
- **Leakage check confirmed:** Train's maximum `TransactionDT` (12,192,900) is strictly less than Test's minimum `TransactionDT` (12,192,911) — an 11-second gap with zero overlap. This proves the split is genuinely time-respecting, not just approximately so.
- **Conclusion:** the split is verified clean and ready to serve as the foundation for Phase 5's model training and evaluation.

## 4.2 Phase 4 Saving and Summary

**Objective recap:** split the Phase 2 engineered feature dataset into training and validation sets using a strict time-based cutoff, avoiding the data leakage a random split would introduce given this dataset's time-ordered structure and rolling window-function features.

**Result:**
- **Train set:** 472,433 rows (`TransactionDT` ≤ 12,192,900) — saved to `data/processed/train_set.csv`
- **Test set:** 118,107 rows (`TransactionDT` ≥ 12,192,911) — saved to `data/processed/test_set.csv`
- **Leakage check:** confirmed zero overlap between the two sets — an 11-second gap separates the latest training transaction from the earliest validation transaction.

**Why this matters:** this split ensures Phase 5's model evaluation genuinely reflects how the model would perform in real deployment — trained only on the past, tested only on the future, with no possibility of the model having implicitly "seen" validation-period patterns during training.

In [8]:
train_df.to_csv("../data/processed/train_set.csv", index=False)
test_df.to_csv("../data/processed/test_set.csv", index=False)

print("Train set saved:", train_df.shape)
print("Test set saved:", test_df.shape)

Train set saved: (472433, 14)
Test set saved: (118107, 14)


**Phase 4 status:** ✅ Complete.